In [8]:
import pandas as pd

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str):
    print(title)
    # 打印列名
    header = " " * 10 + "".join(c.rjust(10) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(10)
        for val in row:
            row_str += colorize(val, width=10)
        print(row_str)
    print()


# ================== 主逻辑 ==================
# 读取两份结果
df4 = pd.read_csv("/mnt/workspace/SELFRec/history_results/results4/result_summary.csv")
df5 = pd.read_csv("/mnt/workspace/SELFRec/history_results/results5/result_summary.csv")

loss_list = [f"loss{i}" for i in range(1, 6)]
topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]
metrics = ["NDCG", "Hit Ratio", "Precision", "Recall"]


# ========== 1. 生成8张表 (平均提升百分比) ==========
def make_tables(df, label):
    for topk in topk_list:
        df_topk = df[df["TopK"] == topk]
        pivot = (
            df_topk[df_topk["Loss"].isin(loss_list)]
            .groupby(["Model", "Loss"])["RelDiff(%)"]
            .mean()
            .reset_index()
        )
        table = pivot.pivot(index="Loss", columns="Model", values="RelDiff(%)").round(3)
        print_colored_table(table, f"\n=== {label} | {topk} ===")

print("========== 汇总表 ==========")
make_tables(df4, "Results4")
make_tables(df5, "Results5")


# ========== 2. 各指标相对提升对比 (百分比) ==========
print("\n========== 各指标结果对比 (Result4 vs Result5 提升百分比) ==========")

merged = pd.merge(
    df4,
    df5,
    on=["Model", "Campus", "Loss", "TopK", "Metric"],
    suffixes=("_r4", "_r5"),
)

for metric in metrics:
    print(f"\n===== Metric: {metric} =====")
    metric_df = merged[merged["Metric"] == metric]
    for topk in topk_list:
        df_topk = metric_df[metric_df["TopK"] == topk]
        pivot = (
            df_topk[df_topk["Loss"].isin(loss_list)]
            .groupby(["Model", "Loss"])[["Value_r4", "Value_r5"]]
            .mean()
            .reset_index()
        )
        pivot["Diff(%)"] = ((pivot["Value_r4"] - pivot["Value_r5"]) / pivot["Value_r5"] * 100).round(3)
        table = pivot.pivot(index="Loss", columns="Model", values="Diff(%)")
        print_colored_table(table, f"\n--- TopK = {topk} ---")


========== 汇总表 ==========

=== Results4 | Top 3 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -1.579    -1.811    -1.700    -1.628
loss2         -2.181    -1.106    -1.284    -1.720
loss3          0.433     0.623    -0.158    -0.538
loss4         -4.909    -5.233    -2.509    -3.979
loss5          1.088    -0.404     0.118    -0.867


=== Results4 | Top 5 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -1.383    -0.951    -0.953    -0.441
loss2         -2.456    -1.771    -0.261    -1.055
loss3         -0.052     0.005     0.202     0.174
loss4         -2.637    -3.569    -1.481    -2.231
loss5          0.604    -0.871     1.533     0.388


=== Results4 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1          0.199     0.749     0.709    -0.013
loss2         -1.082    -0.716     0.184    -0.305
loss3          1.024     1.187     0.079     0.378
loss4         -0.704    -1.891     0.029    -1.226
loss5          1.280    -0